# Codebase-aware dict-seeding — interactive EDA

Companion to `docs/research/2026-06-24-codebase-aware-dict-seeding-eda.md`.
Runs the corpus-analysis engine (`tuparles.nlp`) on **TuParles itself** so it
is reproducible without any other checkout. Point `REPOS` at your own folders
to explore further.

Needs the light `nlp` group (markdown-it / scikit-learn / yake). The final
embedding cell additionally needs `poetry install --with embed` (fastembed).

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent  # notebooks/ -> repo root
sys.path.insert(0, str(REPO / "src"))

from tuparles.nlp import Corpus, code_documents
from tuparles.nlp.engines import dictseed, keywords

REPOS = {"TuParles": REPO}
docs, disc = code_documents(REPOS)
corpus = Corpus()
corpus.ingest(docs)
corpus.finalize()
corpus.compute_metafeatures()
print(f"{disc.mineable} mineable / {disc.n_files} tracked; dropped {dict(disc.skipped)}")
print(f"{len(corpus.stats):,} terms, {len(corpus.candidates()):,} candidates")

## Where salience comes from (the hierarchical weight table at work)

In [ ]:
from collections import Counter
from tuparles.nlp.parse import WEIGHT, SrcType

by_src = Counter()
for ts in corpus.stats.values():
    for stype, n in ts.by_type.items():
        by_src[stype] += n * WEIGHT[SrcType(stype)]
for stype, val in by_src.most_common():
    print(f"{stype:16s} {val:10.0f}")

## Top dict-seed candidates (symbol + TF-IDF, gated by whisper-risk)

In [ ]:
for s in dictseed.seed(corpus, top=20):
    print(f"{s.seed_score:.4f}  risk={s.whisper_risk:.2f}  sal={s.salience:6.0f}  {s.surface}")

## Signal disagreement: salience vs distinctiveness

The key finding — raw salience surfaces frequent-but-useless tokens; TF-IDF
surfaces what is characteristic. The seed list above is the RRF fusion of both.

In [ ]:
from tuparles.nlp.signals import rank_symbol, rank_tfidf

cands = corpus.candidates()
surf = {t.key: t.surface for t in cands}
sym, tf = rank_symbol(cands)[:15], rank_tfidf(cands)[:15]
print(f"{'symbol (salience)':28s} tfidf (distinctiveness)")
for a, b in zip(sym, tf):
    print(f"{surf[a]:28s} {surf[b]}")
print(f"\noverlap: {len(set(sym) & set(tf))}/15")

## Metafeature distributions

In [ ]:
import numpy as np

risk = np.array([dictseed.whisper_risk(t) for t in cands])
ident = np.array([t.is_identifier for t in cands])
print(f"mean whisper-risk: identifiers {risk[ident].mean():.2f} vs prose {risk[~ident].mean():.2f}")
hist, edges = np.histogram(risk, bins=5, range=(0, 1))
for i, n in enumerate(hist):
    print(f"{edges[i]:.1f}-{edges[i+1]:.1f}  {'#' * int(40 * n / max(hist.max(),1))} {n}")
# Try matplotlib if available
try:
    import matplotlib.pyplot as plt
    sal = np.array([t.salience for t in cands]); tfv = np.array([t.tfidf for t in cands])
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].hist(risk, bins=20); ax[0].set_title("whisper-risk")
    ax[1].scatter(np.log1p(sal), tfv, s=6, alpha=.3, c=risk, cmap="viridis")
    ax[1].set_xlabel("log(salience)"); ax[1].set_ylabel("tfidf"); ax[1].set_title("salience vs tfidf (color=risk)")
    plt.tight_layout(); plt.show()
except ImportError:
    print("(matplotlib not installed — ASCII histogram above)")

## Keywords (YAKE) over a prose doc — the non-code path

Same engine, prose source: keyphrases from the README. The clustering and
embedding-keyphrase cells need `poetry install --with embed`.

In [ ]:
readme = (REPO / "README.md").read_text(encoding="utf-8")
for phrase, score in keywords.yake_keyphrases(readme, lang="en", top=12):
    print(f"{score:.4f}  {phrase}")

In [ ]:
# Optional: embeddings (needs `poetry install --with embed`)
try:
    from tuparles.nlp.signals import FastEmbedBackend
    from tuparles.nlp.engines import cluster
    backend = FastEmbedBackend()
    for c in cluster.cluster_terms(corpus, backend, n_clusters=8, min_count=2):
        print(f"[{c.size:3d}] {', '.join(c.label_terms)}")
except ImportError as e:
    print(f"embed group not installed: {e}")